# UVG-VCM 라벨 구조 확인
- Input: 데이터셋 구축 노트북(`uvg-build-dataset` 등) Output 추가
- Accelerator None, Internet Off 가능
- 끝나면 Logs + Output의 `overlay_*.png` 3장 전달

In [ ]:
import os, json, glob, collections
from PIL import Image, ImageDraw
cands = glob.glob("/kaggle/input/**/uvg_cctv", recursive=True)
assert cands, "Input에 uvg_cctv 없음 -> 구축 노트북 Output을 Add Input"
ROOT = cands[0]; print("ROOT", ROOT)
OUT = "/kaggle/working"; SCALE = 3.0

def short(o, n=700):
    s = json.dumps(o, ensure_ascii=False); return s[:n] + ("..." if len(s) > n else "")

for seq in sorted(os.listdir(ROOT)):
    p = os.path.join(ROOT, seq, "ann_raw.json")
    if not os.path.exists(p): continue
    raw = json.load(open(p))
    keys = [k for k in raw if k != "version"]
    nums = sorted(int(k) for k in keys if k.isdigit())
    print(f"\n===== {seq}: version={raw.get('version')} | 프레임 키 {len(keys)}개, 숫자 범위 {nums[0] if nums else None}~{nums[-1] if nums else None}")
    print("  비숫자 키:", [k for k in keys if not k.isdigit()][:10])
    first = raw[str(nums[0])] if nums else None
    print("  첫 프레임 타입:", type(first).__name__, "| 내용:", short(first))
    # 객체 리스트 찾기
    objs = first if isinstance(first, list) else next((v for v in first.values() if isinstance(v, list)), []) if isinstance(first, dict) else []
    if objs:
        kc = collections.Counter(k for o in objs if isinstance(o, dict) for k in o)
        print("  객체 키:", dict(kc))
    # 전체 객체 수 / 카테고리 분포
    tot = 0; cats = collections.Counter()
    for k in keys:
        v = raw[k]
        lst = v if isinstance(v, list) else next((x for x in v.values() if isinstance(x, list)), []) if isinstance(v, dict) else []
        tot += len(lst)
        for o in lst:
            if isinstance(o, dict):
                for ck in ("category", "category_name", "label", "class", "name", "category_id", "class_id"):
                    if ck in o: cats[o[ck]] += 1; break
    print("  전체 객체 수:", tot, "| 클래스 상위:", cats.most_common(8))

In [ ]:
def boxes_of(obj):
    for k in ("bbox", "box", "bounding_box", "xywh", "xyxy"):
        if k in obj and isinstance(obj[k], (list, tuple)) and len(obj[k]) >= 4:
            return [float(v) for v in obj[k][:4]], k
    if all(k in obj for k in ("x", "y", "w", "h")): return [obj["x"], obj["y"], obj["w"], obj["h"]], "x,y,w,h"
    if all(k in obj for k in ("x1", "y1", "x2", "y2")): return [obj["x1"], obj["y1"], obj["x2"], obj["y2"]], "x1..y2"
    return None, None

def draw(seq, key):
    raw = json.load(open(os.path.join(ROOT, seq, "ann_raw.json")))
    v = raw[key]
    objs = v if isinstance(v, list) else next((x for x in v.values() if isinstance(x, list)), [])
    im = Image.open(os.path.join(ROOT, seq, "first.png")).convert("RGB"); d = ImageDraw.Draw(im)
    for o in objs:
        b, how = boxes_of(o)
        if not b: continue
        x, y, a, c = [t / SCALE for t in b]
        d.rectangle([x, y, x + a, y + c], outline=(0, 255, 0), width=2)   # 초록: (x, y, w, h) 해석
        if a > x and c > y:
            d.rectangle([x, y, a, c], outline=(255, 0, 0), width=1)       # 빨강: (x1, y1, x2, y2) 해석
    im.save(f"{OUT}/overlay_{seq}_key{key}.png")
    print(seq, "key", key, "objects", len(objs), "box field:", how if objs else None)

for seq in ["HighwayView", "JobFair", "VolleyballGame"]:
    raw = json.load(open(os.path.join(ROOT, seq, "ann_raw.json")))
    nums = sorted(int(k) for k in raw if k.isdigit())
    draw(seq, str(nums[0]))   # 첫 프레임 키를 기준 영상 첫 프레임(원본 0번)에 그림